In [1]:
import sys
sys.path.append("../..")   # the notebook is 2 folders deep; this points at the repo root

import os
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from src.utils import set_seed, get_device
from src.train import train_model, save_history, load_history

device = get_device()
print("device:", device)



device: cpu


In [2]:
set_seed(42)
X_train, y_train = torch.randn(256, 3, 32, 32), torch.randint(0, 10, (256,))
X_val, y_val = torch.randn(64, 3, 32, 32), torch.randint(0, 10, (64,))
fake_train = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
fake_val = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

def make_stand_in():
    return nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 10))   # replaced by B's MLP later

In [3]:
set_seed(42)
hist = train_model(make_stand_in(), fake_train, fake_val, epochs=2,
                   device=device, save_path="test_ckpt.pt")
assert set(hist) == {"train_loss", "train_acc", "val_loss", "val_acc", "best_epoch", "train_time_s"}
assert all(len(hist[k]) == 2 for k in ["train_loss", "train_acc", "val_loss", "val_acc"])
assert hist["best_epoch"] in (1, 2) and os.path.exists("test_ckpt.pt")
print("history contract OK")

Epoch  1/2 | train loss 2.5200 acc 0.0938 | val loss 2.4276 acc 0.1406 <- best so far
Epoch  2/2 | train loss 1.0176 acc 0.8359 | val loss 2.4408 acc 0.1875
history contract OK


In [4]:
set_seed(42)
hist = train_model(make_stand_in(), fake_train, fake_val, epochs=20,
                   device=device, save_path="test_ckpt.pt", verbose=False)
print("train acc: first %.2f -> last %.2f" % (hist["train_acc"][0], hist["train_acc"][-1]))
print("val acc:   first %.2f -> last %.2f" % (hist["val_acc"][0], hist["val_acc"][-1]))

train acc: first 0.09 -> last 1.00
val acc:   first 0.14 -> last 0.12


In [5]:
set_seed(42)
h1 = train_model(make_stand_in(), fake_train, fake_val, epochs=3, device=device,
                 save_path="test_ckpt.pt", verbose=False)
set_seed(42)
h2 = train_model(make_stand_in(), fake_train, fake_val, epochs=3, device=device,
                 save_path="test_ckpt.pt", verbose=False)
diff = max(abs(a - b) for a, b in zip(h1["train_loss"], h2["train_loss"]))
print("largest difference:", diff)
assert diff < 1e-4, "runs are not repeatable: check set_seed"

largest difference: 0.0


In [6]:
save_history(h1, "test_history.json")
assert load_history("test_history.json")["best_epoch"] == h1["best_epoch"]

m = make_stand_in()
m.load_state_dict(torch.load("test_ckpt.pt", map_location="cpu"))
print("history JSON and checkpoint reload OK")

os.remove("test_ckpt.pt")
os.remove("test_history.json")

history JSON and checkpoint reload OK


In [6]:
save_history(h1, "test_history.json")
assert load_history("test_history.json")["best_epoch"] == h1["best_epoch"]

m = make_stand_in()
m.load_state_dict(torch.load("test_ckpt.pt", map_location="cpu"))
print("history JSON and checkpoint reload OK")

os.remove("test_ckpt.pt")
os.remove("test_history.json")

history JSON and checkpoint reload OK


In [2]:
import sys
sys.path.append("../..")   # point at the repo root so "src" can be found

from src.utils import set_seed, get_device
from src.train import train_model
from src.data import get_dataloaders
from src.model import MLP, count_parameters

device = get_device()
set_seed(42)
train_loader, val_loader, _ = get_dataloaders(data_dir="../../data")   # _ = test set, NOT used
model = MLP()
print("parameters:", count_parameters(model))

hist = train_model(model, train_loader, val_loader, epochs=5, lr=1e-3,
                   device=device, save_path="day2_test.pt")
print("time per epoch: %.0f s" % (hist["train_time_s"] / 5))

parameters: 1707274
Epoch  1/5 | train loss 1.6570 acc 0.4118 | val loss 1.5713 acc 0.4368 <- best so far
Epoch  2/5 | train loss 1.4450 acc 0.4899 | val loss 1.5042 acc 0.4744 <- best so far
Epoch  3/5 | train loss 1.3380 acc 0.5295 | val loss 1.4664 acc 0.4914 <- best so far
Epoch  4/5 | train loss 1.2427 acc 0.5622 | val loss 1.4390 acc 0.4994 <- best so far
Epoch  5/5 | train loss 1.1629 acc 0.5889 | val loss 1.4182 acc 0.5150 <- best so far
time per epoch: 57 s


In [ ]:
import os
os.remove("day2_test.pt")
print("deleted")